# 03 — XGBoost Demand Forecasting

**Reference:** Vandeput, N. (2021). *Data Science for Supply Chain Forecasting* (2nd ed.), Chapter 21.

XGBoost (Extreme Gradient Boosting) won the M5 forecasting competition —
the largest real-world forecasting benchmark ever run. It outperforms
statistical models when:
- Multiple external features are available (weather, events, promotions)
- The demand pattern is non-linear
- You have enough historical data (ideally 2+ years)

**Key idea:** Frame time series forecasting as a supervised ML problem.
Create features from lag values, rolling statistics, and calendar variables.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

## 1. Generate Realistic FMCG Demand Dataset

In [ ]:
np.random.seed(42)

# 5 years of weekly demand
n_weeks = 260
dates = pd.date_range(start='2019-01-01', periods=n_weeks, freq='W')

# Components
trend = 1000 + 1.5 * np.arange(n_weeks)
seasonality = 150 * np.sin(2 * np.pi * np.arange(n_weeks) / 52)  # yearly cycle
noise = np.random.normal(0, 60, n_weeks)

demand = trend + seasonality + noise
demand = np.clip(demand, 0, None)

df = pd.DataFrame({'date': dates, 'demand': demand})
df = df.set_index('date')
print(df.describe())

## 2. Feature Engineering — The Core of ML Forecasting

In [ ]:
def create_features(df, label='demand', lags=[1,2,3,4,8,12,26,52]):
    """
    Create time series features for supervised ML forecasting.

    Features created:
    - Lag features: demand at t-1, t-2, ..., t-52
    - Rolling statistics: mean and std over recent windows
    - Calendar features: week of year, month, quarter
    """
    df = df.copy()

    # Lag features
    for lag in lags:
        df[f'lag_{lag}'] = df[label].shift(lag)

    # Rolling statistics
    df['rolling_mean_4']  = df[label].shift(1).rolling(4).mean()
    df['rolling_mean_12'] = df[label].shift(1).rolling(12).mean()
    df['rolling_std_4']   = df[label].shift(1).rolling(4).std()

    # Calendar features
    df['week_of_year'] = df.index.isocalendar().week.astype(int)
    df['month']        = df.index.month
    df['quarter']      = df.index.quarter

    return df


df_features = create_features(df)
df_features = df_features.dropna()  # remove rows with NaN lag values

print(f'Features created: {df_features.shape[1] - 1} input features')
print(f'Training samples: {len(df_features)}')
print(df_features.columns.tolist())

## 3. Train/Test Split — Time Series Style

Critical: never use random splits for time series. Always split chronologically.

In [ ]:
FORECAST_HORIZON = 12  # 12-week holdout

feature_cols = [c for c in df_features.columns if c != 'demand']
X = df_features[feature_cols]
y = df_features['demand']

X_train, X_test = X.iloc[:-FORECAST_HORIZON], X.iloc[-FORECAST_HORIZON:]
y_train, y_test = y.iloc[:-FORECAST_HORIZON], y.iloc[-FORECAST_HORIZON:]

print(f'Train: {len(X_train)} weeks | Test: {len(X_test)} weeks')

## 4. Train XGBoost Model

In [ ]:
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=30,
    eval_metric='mae'
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f'Test MAE: {mae:.1f} units')
print(f'Test MAPE: {(abs(y_test.values - y_pred) / y_test.values).mean()*100:.1f}%')

## 5. Feature Importance — What Drives Demand?

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind='barh', ax=ax, color='#185FA5', alpha=0.8)
ax.set_title('XGBoost Feature Importance — Demand Forecasting', fontsize=13)
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Forecast vs Actual

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

# Show last 52 weeks of training + full test period
ax.plot(y_train.iloc[-52:], label='Training data', color='#888780', linewidth=1.2)
ax.plot(y_test, label='Actual demand', color='#2C2C2A', linewidth=1.5)
ax.plot(y_test.index, y_pred, label='XGBoost forecast',
        color='#185FA5', linestyle='--', linewidth=1.5)
ax.axvline(x=y_test.index[0], color='gray', linestyle=':', alpha=0.6)
ax.set_title('XGBoost Demand Forecasting — 12-Week Holdout Evaluation', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Weekly Demand (units)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

- XGBoost requires feature engineering — the quality of your features determines accuracy
- Always use chronological train/test splits for time series
- Lag features are the most important signal — `lag_1` (last week's demand) usually dominates
- External features (weather, events) can dramatically improve accuracy — see next notebook

**Next:** External demand drivers (`04_external_demand_drivers.ipynb`) — the key differentiator.
